# KubeHeal AI - Exploratory Data Analysis
## Kubernetes Failure Prediction Dataset Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

In [ ]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = list(processed_dir.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files')

if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f'Loaded: {csv_files[0].name}')
    print(f'Shape: {df.shape}')
else:
    print('Generating synthetic dataset for EDA...')
    np.random.seed(42)
    n = 5000
    df = pd.DataFrame({
        'cpu_usage': np.random.uniform(0, 100, n),
        'memory_usage': np.random.uniform(0, 100, n),
        'disk_usage': np.random.uniform(0, 100, n),
        'network_errors': np.random.poisson(2, n),
        'restart_count': np.random.poisson(1, n),
        'latency_ms': np.random.exponential(100, n),
        'severity': np.random.choice(['info','warning','critical','ok'], n),
        'pod_status': np.random.choice(['Running','Pending','Failed','CrashLoopBackOff'], n, p=[0.6,0.15,0.12,0.13]),
    })
    score = (df['cpu_usage'] > 85).astype(int) + (df['memory_usage'] > 85).astype(int) + (df['restart_count'] > 3).astype(int) * 2
    df['failure'] = (score >= 2).astype(int)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
numeric_cols = ['cpu_usage', 'memory_usage', 'disk_usage', 'network_errors', 'restart_count', 'latency_ms']
for ax, col in zip(axes.flat, numeric_cols):
    df[col].hist(bins=50, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
if 'failure' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.boxplot(x='failure', y='cpu_usage', data=df, ax=axes[0])
    sns.boxplot(x='failure', y='memory_usage', data=df, ax=axes[1])
    sns.boxplot(x='failure', y='restart_count', data=df, ax=axes[2])
    axes[0].set_title('CPU Usage by Failure Status')
    axes[1].set_title('Memory Usage by Failure Status')
    axes[2].set_title('Restart Count by Failure Status')
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
if 'pod_status' in df.columns:
    plt.figure(figsize=(10, 5))
    df['pod_status'].value_counts().plot(kind='bar')
    plt.title('Pod Status Distribution')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
print('=== Insights ===')
print(f'Total samples: {len(df)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicated rows: {df.duplicated().sum()}')
if 'failure' in df.columns:
    print(f'Failure rate: {df["failure"].mean():.2%}')
    print(f'Class balance: {df["failure"].value_counts().to_dict()}')

In [ ]:
output_path = processed_dir / 'final_processed_dataset.csv'
df.to_csv(output_path, index=False)
print(f'Saved processed dataset to {output_path}')